# Установка моделей

In [1]:
from pprint import pprint

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Установка токенизаторов

In [2]:
model_name_1 = "LiquidAI/LFM2.5-230M"
model_name_2 = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(model_name_1,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

tokenizer_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(model_name_2,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1261.55it/s]


# Класс генерации forward pass'а

In [3]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend
from transformers.tokenization_utils_tokenizers import TokenizersBackend

TokenizerType = TokenizersBackend | SentencePieceBackend

class Generation:
    def __init__(
        self,
        llm_1: GenerationMixin,
        llm_2: GenerationMixin,
        tok_1: TokenizerType,
        tok_2: TokenizerType,
        top_k: int,
    ):
        self.llm_1 = llm_1
        self.llm_2 = llm_2
        self.tok_1 = tok_1
        self.tok_2 = tok_2
        self.top_k = top_k
        self.device_1 = next(llm_1.parameters()).device
        self.device_2 = next(llm_2.parameters()).device
        self.chat_prefix_1: str | None = None
        self.chat_prefix_2: str | None = None

    @staticmethod
    def _build_chat_prefix(
        tokenizer: TokenizerType,
        user_message: str,
    ) -> str:
        messages = [
            {
                "role": "user",
                "content": user_message,
            }
        ]

        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    @staticmethod
    def _tokenize_prompt(
        tokenizer: TokenizerType,
        prompt: str,
        device: torch.device,
    ):
        return tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        ).to(device)

    @staticmethod
    def _generate_log_probs(
        model: GenerationMixin,
        inputs,
    ) -> torch.Tensor:
        with torch.inference_mode():
            logits = model(**inputs).logits[:, -1, :]

        return torch.log_softmax(
            logits.float(),
            dim=-1,
        )

    def _get_top_distribution(
        self,
        log_probs: torch.Tensor,
        tokenizer: TokenizerType,
        model_number: int,
    ) -> dict[str, float]:
        k = min(self.top_k, log_probs.shape[-1])

        top_log_probs, top_token_ids = torch.topk(
            log_probs[0],
            k=k,
        )

        distribution: dict[str, float] = {}

        print(f"Модель {model_number}")

        for rank, (token_id, log_prob) in enumerate(
            zip(top_token_ids, top_log_probs, strict=True),
            start=1,
        ):
            token_id_int = token_id.item()

            token_text = tokenizer.decode(
                [token_id_int],
                skip_special_tokens=False,
            )

            probability = log_prob.exp().item()

            print(
                f"{rank}. "
                f"token={token_text!r}, "
                f"id={token_id_int}, "
                f"prob={probability:.6f}"
            )

            distribution[token_text] = (
                distribution.get(token_text, 0.0)
                + probability
            )

        return distribution

    def initialize_chat(self, user_message: str) -> None:
        self.chat_prefix_1 = self._build_chat_prefix(
            tokenizer=self.tok_1,
            user_message=user_message,
        )

        self.chat_prefix_2 = self._build_chat_prefix(
            tokenizer=self.tok_2,
            user_message=user_message,
        )

        print("Конец chat-prefix модели 1:")
        print(repr(self.chat_prefix_1[-150:]))

        print("Конец chat-prefix модели 2:")
        print(repr(self.chat_prefix_2[-150:]))

    def generate_pipe(
        self,
        generated_text: str,
        model_to_run: str = '',
    ):
        if self.chat_prefix_1 is None or self.chat_prefix_2 is None:
            raise RuntimeError(
                "Сначала вызови initialize_chat(user_message)"
            )

        distributions: dict[str, dict[str, float]] = {}
        if model_to_run == "":
            print('попали в условие запуска всех моделей')
            prompt_1 = self.chat_prefix_1 + generated_text

            inputs_1 = self._tokenize_prompt(
                tokenizer=self.tok_1,
                prompt=prompt_1,
                device=self.device_1,
            )

            log_probs_1 = self._generate_log_probs(
                model=self.llm_1,
                inputs=inputs_1,
            )

            distributions["1"] = self._get_top_distribution(
                log_probs=log_probs_1,
                tokenizer=self.tok_1,
                model_number=1,
            )

            prompt_2 = self.chat_prefix_2 + generated_text
            inputs_2 = self._tokenize_prompt(
                tokenizer=self.tok_2,
                prompt=prompt_2,
                device=self.device_2,
            )

            log_probs_2 = self._generate_log_probs(
                model=self.llm_2,
                inputs=inputs_2,
            )

            distributions["2"] = self._get_top_distribution(
                log_probs=log_probs_2,
                tokenizer=self.tok_2,
                model_number=2,
            )

        if model_to_run == "model_0":
            prompt_1 = self.chat_prefix_1 + generated_text
            print('попали в условие запуска модели 0')
            inputs_1 = self._tokenize_prompt(
                tokenizer=self.tok_1,
                prompt=prompt_1,
                device=self.device_1,
            )

            log_probs_1 = self._generate_log_probs(
                model=self.llm_1,
                inputs=inputs_1,
            )

            distributions["1"] = self._get_top_distribution(
                log_probs=log_probs_1,
                tokenizer=self.tok_1,
                model_number=1,
            )

        if model_to_run == "model_1":
            prompt_2 = self.chat_prefix_2 + generated_text
            print('попали в условие запуска модели 1')
            inputs_2 = self._tokenize_prompt(
                tokenizer=self.tok_2,
                prompt=prompt_2,
                device=self.device_2,
            )

            log_probs_2 = self._generate_log_probs(
                model=self.llm_2,
                inputs=inputs_2,
            )

            distributions["2"] = self._get_top_distribution(
                log_probs=log_probs_2,
                tokenizer=self.tok_2,
                model_number=2,
            )

        if "1" in distributions and "2" in distributions:
            return distributions["1"], distributions["2"]

        if "1" in distributions:
            return distributions["1"]

        if "2" in distributions:
            return distributions["2"]

        raise RuntimeError("Не была запущена ни одна модель")


# Класс префиксной плотности

In [ ]:
class PrefixDense:
    def __init__(self,
                 probs_generator: Generation,
                 input_str: str,
                 stop_token: str):
        self.prob_distribution_1 = {}
        self.prob_distribution_2 = {}
        self.step_matrix = {}
        self.model_to_run = ''
        self.max_steps: int = 10
        self.probs_generator = probs_generator
        self.user_prompt = input_str
        self.generated_text = ""
        self.max_steps = 512
        self.stop_token = stop_token

    
    def runpipe(self):
        self.probs_generator.initialize_chat(
            user_message=self.user_prompt,
        )
        print(f'{self.max_steps}')
        for step in range(self.max_steps):
            print("###############################################################")
            print(f"Шаг: {step}")
            print("Пользовательский запрос:")
            print(self.user_prompt)
            print("Сгенерированное продолжение:")
            print(repr(self.generated_text))
            print("###############################################################")
            result = self.probs_generator.generate_pipe(
                generated_text=self.generated_text,
                model_to_run=self.model_to_run,
            )
            self.prob_distribution_1, self.prob_distribution_2 = result
            print(f'{self.prob_distribution_1=}')
            print(f'{self.prob_distribution_2=}')
            ### Блок сопоставления токенов ###
            selected_text = self.match_chars()
            self.generated_text += selected_text
            print(f"Выбранный фрагмент: {selected_text!r}")
            if self.stop_token in selected_text:
                break
            print(f"Текущий ответ: {self.generated_text!r}")
        return self.generated_text

    def count_min_token_len_per_distrib(self, distribs: list):
        distrib_min_len = {}
        for idx, distrib in enumerate(distribs):
            distrib_min_len[idx] = min({len(key) for key in distrib})
        return distrib_min_len

    def count_max_token_len_per_distrib(self, distribs: list):
        distrib_min_len = {}
        for idx, distrib in enumerate(distribs):
            distrib_min_len[idx] = max({len(key) for key in distrib})
        return distrib_min_len

    def find_the_suitest_token(self, 
                               distrib: dict, 
                               char_num: int, 
                               distrib_num: int):
        char_prob = {}
        new_distrib = {}
        print(f'Для теста {distrib=}')
        for token, probability in distrib.items():
            print(f"{char_num=}")
            print(f"{token=}")
            if char_num >= len(token):
                print(
                    f'Индекс {char_num} выходит за границы '
                    f'токена "{token}".'
                )
                result = self.probs_generator.generate_pipe(
                    generated_text=self.generated_text + token,
                    model_to_run=f"model_{distrib_num}",
                )
                if isinstance(result, (tuple, list)):
                    result = result[distrib_num]
                pprint(
                    f'Для токена "{token}" с вероятностью '
                    f"{probability} получили {result=}"
                )
                for token2, prob2 in result.items():
                    if not token2 or prob2 <= 0:
                        continue
                    ongoing_token = token + token2
                    ongoing_prob = probability * prob2
                    print(
                        f'Полученный токен "{ongoing_token}", '
                        f"вероятность={ongoing_prob}"
                    )
                    new_distrib[ongoing_token] = (
                        new_distrib.get(ongoing_token, 0.0)
                        + ongoing_prob
                    )
                    if char_num < len(ongoing_token):
                        current_char = ongoing_token[char_num]

                        char_prob[current_char] = (
                            char_prob.get(current_char, 0.0)
                            + ongoing_prob
                        )
            else:
                new_distrib[token] = (
                    new_distrib.get(token, 0.0)
                    + probability
                )
                current_char = token[char_num]
                char_prob[current_char] = (
                    char_prob.get(current_char, 0.0)
                    + probability
                )
            tmp_dict = {}
            for token, prob in char_prob.items():
                print(f'Подсчитываем {token} — {prob}')
                print(f'Сумма по всему вероятностному распределению — {sum(distrib.values())}')
                tmp_dict[token] = prob/sum(distrib.values())
            print(f"После пересчета {tmp_dict}")

        if not char_prob:
            pass
        print(f"Первоначально {char_prob}")
        print(f"{new_distrib=}")
        the_most_common_char = max(
            char_prob,
            key=char_prob.get,
        ) if char_prob else ''
        return the_most_common_char, new_distrib
    
    def ensemble(self, ensemble_distr: dict):
        token_prob = {}
        for distrib in ensemble_distr.values():
            for token, prob in distrib.items():
                if token not in token_prob:
                    token_prob[token] = prob
                else:
                    token_prob[token] += prob
        
        the_most_popular_token = max(token_prob, key=token_prob.get)
        return the_most_popular_token

    def ensemble_str(self, ensemble_dict: dict, char_num: int, probs_list: list):
        ensembling_probs = {}
        print(f'здесь {ensemble_dict=}')
        for distr in ensemble_dict.values():
            print(f'{distr=}')
            for distribution in distr.values():
                if not isinstance(distribution, str):
                    for token, prob in distribution.items():
                        if token not in ensembling_probs:
                            ensembling_probs[token] = prob
                        else:
                            ensembling_probs[token] += prob
                else:
                    result = "".join(distr[key] for key in sorted(distr))
                    print(f'{result=}')
                    print(f'{probs_list=}')
                    for elem in probs_list:
                        for token, prob in elem.items():
                            if token.startswith(result):
                                print(f'Токен {token} начинается с {result}, добавляем его вероятность')
                                if result not in ensembling_probs:
                                    ensembling_probs[result] = prob
                                else:
                                    ensembling_probs[result] += prob        
        pprint(f'{ensembling_probs=}')
        most_likely_token = max(ensembling_probs, key=ensembling_probs.get)
        return most_likely_token
   

    def match_chars(self):
        probs_list = [self.prob_distribution_1, self.prob_distribution_2]
        distrib_min_len = self.count_min_token_len_per_distrib(distribs=probs_list)
        distrib_max_len = self.count_max_token_len_per_distrib(distribs=probs_list)

        ensemble_dict = {}
        print(f'{probs_list=}')
        print(f'{distrib_min_len=}')
        print(f'{distrib_max_len=}')
        prefix = ''
        max_char_steps = 128
        char_num = 0
        while char_num < max_char_steps:
            print(f'Внимание! {char_num=}')
            print(f'Итерируемся по такому распределению {probs_list=}')
            for_suitest = []
            for distrib_num, distrib in enumerate(probs_list):
                print(f'До раскрытия: {probs_list}')
                the_most_common_char, new_distrib = self.find_the_suitest_token(distrib=distrib, char_num=char_num, distrib_num=distrib_num) 
                print(f'После раскрытия  {new_distrib=}')
                for_suitest.append(new_distrib)
                # the_most_common_char, updated_distrib = self.find_the_suitest_token(distrib=distrib, char_num=char_num, distrib_num=distrib_num)
                print(f'{the_most_common_char=}')
                print(f'Ансамбль дикт на старте {ensemble_dict=}')
                if isinstance(the_most_common_char, dict): # Кейс, когда нет общих совпадений внутри распределений
                    print('перезаписываем1')
                    ensemble_dict[f'distrib_num_{distrib_num}'] = {char_num: the_most_common_char}
                elif isinstance(the_most_common_char, str): # Кейс, когда есть общие совпадения внутри распределений
                    if f'distrib_num_{distrib_num}' not in ensemble_dict:
                        print('перезаписываем')
                        ensemble_dict[f'distrib_num_{distrib_num}'] = {char_num: the_most_common_char}
                    else:
                        ensemble_dict[f'distrib_num_{distrib_num}'][char_num] = the_most_common_char

            print(f'Ансамбль дикт на финише {ensemble_dict=}')
            print(f'Финалисты {char_num}-того символа {ensemble_dict=}')
            probs_list = for_suitest
            most_likely_token = self.ensemble_str(ensemble_dict=ensemble_dict, char_num=char_num, probs_list=probs_list)
            prefix = prefix + most_likely_token[-1]
            print(f'{prefix=}')
            print(f'{most_likely_token=}')
            print(f'{probs_list=}')
            new_probs_list = []
            
            # Сортировка по релеватным для продолжения токенам
            print(f'Начинаем зачистку токенов, не начинающихся с "{prefix}"')
            pprint(f'Как было до: {probs_list=}')
            for elem in probs_list:
                tmp_distrib = {}
                for token, prob in elem.items():
                    if token.startswith(most_likely_token):
                        print(f'Токен "{token}" начинается с "{most_likely_token}", оставляем его в пуле на следующую проверку')
                        tmp_distrib[token] = prob
                new_probs_list.append(tmp_distrib)
            
            print(f'Как стало после: {new_probs_list=}')
            if all(current.keys() == new_probs_list[0].keys() for current in new_probs_list[1:]):
                summed_probs = {
                    key: sum(distrib[key] for distrib in new_probs_list)
                    for key in new_probs_list[0]
                }

                return_string = max(summed_probs, key=summed_probs.get)

                print(f'Суммарные вероятности: {summed_probs}')
                print(f'Возвращаем {return_string}')

                return return_string

            for distrib in new_probs_list:
                print(len(distrib))
                if len(distrib) == 1:
                    for string in distrib:
                        return_string = string
                    return return_string

            char_num += 1
            probs_list = new_probs_list          

In [5]:
agg = Generation(
    llm_1=model_1,
    llm_2=model_2,
    tok_1=tokenizer_1,
    tok_2=tokenizer_2,
    top_k=5,
)

agreement = PrefixDense(
    probs_generator=agg,
    input_str='In TCP/IP networking, which protocol is used to hold network addresses and routing information in a packet? "A": "HTTP", "B": "IP", "C": "Routing Information Protocol (RIP)", "D": "TCP"',
    stop_token='<|im_end|>'
)

answer = agreement.runpipe()

print("Итог:")
print(repr(answer))

Конец chat-prefix модели 1:
'es and routing information in a packet? "A": "HTTP", "B": "IP", "C": "Routing Information Protocol (RIP)", "D": "TCP"<|im_end|>\n<|im_start|>assistant\n'
Конец chat-prefix модели 2:
'es and routing information in a packet? "A": "HTTP", "B": "IP", "C": "Routing Information Protocol (RIP)", "D": "TCP"<|im_end|>\n<|im_start|>assistant\n'
512
###############################################################
Шаг: 0
Пользовательский запрос:
In TCP/IP networking, which protocol is used to hold network addresses and routing information in a packet? "A": "HTTP", "B": "IP", "C": "Routing Information Protocol (RIP)", "D": "TCP"
Сгенерированное продолжение:
''
###############################################################
попали в условие запуска всех моделей
Модель 1
1. token='The', id=1098, prob=0.977938
2. token='**', id=1463, prob=0.003814
3. token='Correct', id=64008, prob=0.002924
4. token='B', id=543, prob=0.002747
5. token='To', id=3097, prob=0.002139
Модель 2
1

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="bfloat16",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

prompt = 'In TCP/IP networking, which protocol is used to hold network addresses and routing information in a packet? "A": "HTTP", "B": "IP", "C": "Routing Information Protocol (RIP)", "D": "TCP"'

input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
)["input_ids"].to(model.device)

output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.1,
    top_k=50,
    repetition_penalty=1.05,
    max_new_tokens=512,
    streamer=streamer,
)